In [0]:

CREATE SCHEMA IF NOT EXISTS enterprise.silver_ingestion;

CREATE TABLE IF NOT EXISTS enterprise.silver_ingestion.gold_crypto_features_1m (
  source        STRING,
  symbol        STRING,
  bar_start_ts  TIMESTAMP,
  bar_end_ts    TIMESTAMP,
  open          DOUBLE,
  high          DOUBLE,
  low           DOUBLE,
  close         DOUBLE,
  volume        DOUBLE,
  ingest_ts     TIMESTAMP,
  p_date        DATE,
  prev_close    DOUBLE,
  candle_body   DOUBLE,
  candle_range  DOUBLE,
  ret_1m        DOUBLE,
  log_ret_1m    DOUBLE,
  ma_close_15m  DOUBLE,
  ma_close_60m  DOUBLE,
  vol_logret_60m DOUBLE,
  updated_at    TIMESTAMP
)
USING DELTA
PARTITIONED BY (p_date)
COMMENT 'Gold Layer: Financial features (MA, Returns, Volatility) derived from Silver inputs';


MERGE INTO enterprise.silver_ingestion.gold_crypto_features_1m AS tgt
USING (
  WITH base AS (
    SELECT
      source,
      symbol,
      bar_start_ts,
      bar_end_ts,
      open,
      high,
      low,
      close,
      volume,
      p_date,
      ingest_ts
    FROM enterprise.silver_ingestion.slv_crypto_ohlc_1m
    -- Optional: filter for incremental run here if needed, e.g. p_date >= current_date() - 1
  ),
  features AS (
    SELECT
      *,
      -- Previous Close
      LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) AS prev_close,

      -- Candle Features
      (close - open) AS candle_body,
      (high - low)   AS candle_range,

      -- Simple Return (1m)
      -- Handle division by zero or null previous close
      CASE 
        WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) IS NULL THEN NULL
        WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) = 0 THEN 0
        ELSE (close / LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts)) - 1
      END AS ret_1m,

      -- Log Return (1m)
      -- Handle non-positive prices for log calculation
      CASE
        WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) <= 0 OR close <= 0 THEN 0
        WHEN LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts) IS NULL THEN NULL
        ELSE LN(close / LAG(close) OVER (PARTITION BY source, symbol ORDER BY bar_start_ts))
      END AS log_ret_1m

    FROM base
  ),
  rolling AS (
    SELECT
      *,
      -- Moving Averages (15m, 60m)
      AVG(close) OVER (
        PARTITION BY source, symbol 
        ORDER BY bar_start_ts 
        ROWS BETWEEN 14 PRECEDING AND CURRENT ROW
      ) AS ma_close_15m,

      AVG(close) OVER (
        PARTITION BY source, symbol 
        ORDER BY bar_start_ts 
        ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
      ) AS ma_close_60m,

      -- Volatility (StdDev of Log Returns over 60m)
      STDDEV(log_ret_1m) OVER (
        PARTITION BY source, symbol 
        ORDER BY bar_start_ts 
        ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
      ) AS vol_logret_60m

    FROM features
  )
  SELECT 
    *,
    current_timestamp() as updated_at 
  FROM rolling
) AS src
ON  tgt.source = src.source
AND tgt.symbol = src.symbol
AND tgt.bar_start_ts = src.bar_start_ts

WHEN MATCHED THEN UPDATE SET
  tgt.bar_end_ts     = src.bar_end_ts,
  tgt.open           = src.open,
  tgt.high           = src.high,
  tgt.low            = src.low,
  tgt.close          = src.close,
  tgt.volume         = src.volume,
  tgt.ingest_ts      = src.ingest_ts,
  tgt.p_date         = src.p_date,
  tgt.prev_close     = src.prev_close,
  tgt.candle_body    = src.candle_body,
  tgt.candle_range   = src.candle_range,
  tgt.ret_1m         = src.ret_1m,
  tgt.log_ret_1m     = src.log_ret_1m,
  tgt.ma_close_15m   = src.ma_close_15m,
  tgt.ma_close_60m   = src.ma_close_60m,
  tgt.vol_logret_60m = src.vol_logret_60m,
  tgt.updated_at     = src.updated_at

WHEN NOT MATCHED THEN INSERT (
  source, symbol, bar_start_ts, bar_end_ts, open, high, low, close, volume, 
  ingest_ts, p_date, prev_close, candle_body, candle_range, 
  ret_1m, log_ret_1m, ma_close_15m, ma_close_60m, vol_logret_60m, updated_at
) VALUES (
  src.source, src.symbol, src.bar_start_ts, src.bar_end_ts, src.open, src.high, src.low, src.close, src.volume,
  src.ingest_ts, src.p_date, src.prev_close, src.candle_body, src.candle_range,
  src.ret_1m, src.log_ret_1m, src.ma_close_15m, src.ma_close_60m, src.vol_logret_60m, src.updated_at
);

-- COMMAND ----------

-- Verification Query
SELECT * 
FROM enterprise.silver_ingestion.gold_crypto_features_1m 
ORDER BY bar_start_ts DESC 
LIMIT 20;
